# Emotion Mapping for Song Lyrics

This notebook maps emotions to cleaned song lyrics using RoBERTa-base fine-tuned on GoEmotions dataset, then converts them to 6 FER (Facial Emotion Recognition) categories.

## Process:
1. Load cleaned lyrics dataset from Kaggle
2. Use pretrained RoBERTa model to predict 28 GoEmotions probabilities per line
3. Map 28 GoEmotions into 6 FER categories (Angry, Fear, Happy, Sad, Surprise, Neutral)
4. Normalize the FER emotion vector so it sums to 1
5. Compute emotional weight per line: w_i = 1 - Neutral_i
6. Aggregate line-level emotions into song-level vector using weighted averaging
7. Determine dominant emotion (argmax) for each song
8. Save results to CSV

## Key Features:
- **GPU Optimized**: Batch processing for maximum GPU utilization
- **Memory Efficient**: Processes 3M+ rows without loading all into memory
- **Resume Capability**: Manually resumes if Colab session disconnects from the next chunks
- **Real-time Progress**: Visual progress bar with chunk tracking
- **Incremental Saving**: Results saved after each chunk (no data loss)

## 1. Install Required Packages

In [ ]:
!pip install -q transformers tqdm


## 2. Import Libraries

In [4]:
import os
import warnings
import logging

# ── Suppress ALL warnings before any other import ────────────────────────────
# Set env vars first — these are picked up by subprocesses and the C-level
# PyTorch / tokenizer code before Python-level filters take effect.
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # HuggingFace tokenizer parallel-fork warning
os.environ["TORCHDYNAMO_VERBOSE"] = "0"          # torch.compile / dynamo verbose tracing
os.environ["PYTHONWARNINGS"] = "ignore"          # subprocess-level warning suppression

warnings.filterwarnings("ignore")               # Python warnings module (UserWarning, FutureWarning, etc.)

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Silence torch._dynamo / inductor / compile loggers at the logging level
for _logger_name in (
    "torch._dynamo",
    "torch._inductor",
    "torch._inductor.compile_fx",
    "torch._inductor.triton_heuristics",
    "torch.compile",
    "torch.fx",
    "transformers",
):
    logging.getLogger(_logger_name).setLevel(logging.ERROR)

# Set device
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


GPU count: 2
0 Tesla T4
1 Tesla T4


### Performance Tips:
- **Batch Size**: Default is 2048. Lower to 256 if OOM (out of memory).
- **Chunk Size**: process 5000 songs at a time to balance memory and speed. Adjust based on your dataset size and GPU capacity.

## 3. Load Pretrained RoBERTa-GoEmotions Model

In [5]:
# Load RoBERTa model fine-tuned on GoEmotions (28 labels: 27 emotions + neutral)
model_name = "SamLowe/roberta-base-go_emotions"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model.eval()

# Get GoEmotions labels
emotion_labels = list(model.module.config.id2label.values())
print(f"Number of GoEmotions: {len(emotion_labels)}")
print(f"GoEmotions: {emotion_labels}")

# Define FER emotion groups (mapping GoEmotions -> 6 FER categories)
FER_MAPPING = {
    'Angry':    ['anger', 'annoyance', 'disapproval', 'disgust'],
    'Fear':     ['fear', 'nervousness'],
    'Happy':    ['admiration', 'amusement', 'approval', 'caring', 'desire',
                 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'],
    'Sad':      ['sadness', 'grief', 'disappointment', 'remorse', 'embarrassment'],
    'Surprise': ['surprise', 'realization'],
    'Neutral':  ['neutral', 'curiosity', 'confusion'],
}

FER_LABELS = list(FER_MAPPING.keys())  # ['Angry', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Build index mapping: for each FER category, store the indices into the 28-dim GoEmotions vector
FER_INDICES = {}
for fer_cat, go_emotions in FER_MAPPING.items():
    indices = [emotion_labels.index(e) for e in go_emotions]
    FER_INDICES[fer_cat] = indices

print(f"\nFER Categories: {FER_LABELS}")
for cat, indices in FER_INDICES.items():
    mapped = [emotion_labels[i] for i in indices]
    print(f"  {cat}: {mapped}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Using 2 GPUs
Number of GoEmotions: 28
GoEmotions: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

FER Categories: ['Angry', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
  Angry: ['anger', 'annoyance', 'disapproval', 'disgust']
  Fear: ['fear', 'nervousness']
  Happy: ['admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']
  Sad: ['sadness', 'grief', 'disappointment', 'remorse', 'embarrassment']
  Surprise: ['surprise', 'realization']
  Neutral: ['neutral', 'curiosity', 'confusion']


## 4 GPU Optimization Settings

In [6]:
# Enable optimizations for faster inference
torch.backends.cudnn.benchmark = True
torch.set_grad_enabled(False)  # Disable gradient computation globally

# ── Silence dynamo/inductor at the config level (belt-and-suspenders) ────────
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.verbose = False

# Disable CUDA graph recording at the inductor level (both known attribute paths)
for _attr in ("triton.cudagraph_trees", "triton.cudagraph_skip_dynamic_graphs",
              "cudagraph_skip_dynamic_graphs"):
    try:
        obj, _, key = _attr.rpartition(".")
        target = torch._inductor.config
        if obj:
            for part in obj.split("."):
                target = getattr(target, part)
        setattr(target, key, True)
    except Exception:
        pass

# ── Mixed Precision (auto-enabled for GPUs with Tensor Cores: T4/V100/A100/RTX) ──
# Uses FP16 for matrix multiplications, keeping FP32 for softmax → ~1.5-2x speedup
from torch.cuda.amp import autocast
USE_MIXED_PRECISION = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7
print(f"Mixed precision (FP16): {'ENABLED' if USE_MIXED_PRECISION else 'DISABLED (GPU too old or CPU-only)'}")

# ── torch.compile (PyTorch 2.0+) ─────────────────────────────────────────────
# mode="default" applies kernel fusion optimizations WITHOUT recording CUDA graphs.
# This is the key fix: mode="reduce-overhead" is what enables CUDA graph recording,
# causing a new graph to be captured for every unique (batch_size, seq_len) shape
# → thousands of "cudagraph partition due to dynamic shape ops" warnings.
# mode="default" + dynamic=True gives a single compiled graph for all shapes, zero warnings.
try:
    model = torch.compile(model, mode="default", dynamic=True, fullgraph=False)
    print("torch.compile: ENABLED (mode=default, dynamic=True)")
except Exception:
    print("torch.compile: SKIPPED (requires PyTorch >= 2.0)")

# Check GPU memory
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print("\nAll GPU optimizations enabled")
else:
    print("No GPU available, using CPU (will be slower)")


Mixed precision (FP16): ENABLED
torch.compile: ENABLED (mode=default, dynamic=True)

GPU: Tesla T4
Total GPU Memory: 15.64 GB
Allocated: 0.50 GB
Cached: 0.56 GB

All GPU optimizations enabled


## 5. Define Emotion Mapping Functions

**Pipeline per line:**
1. Predict GoEmotions probabilities (28-dim vector)
2. Map to FER categories by summing group probabilities (6-dim vector)
3. Normalize so FER scores sum to 1

**Song-level aggregation:**
4. Weight each line: w_i = 1 - Neutral_i
5. Weighted average: E_song = Σ(w_i · q_i) / Σ(w_i)
6. Dominant emotion = argmax(E_song)

In [7]:
from torch.utils.data import DataLoader


def predict_emotions_batch(texts, batch_size=2048, max_length=128):
    """
    Predict GoEmotions probabilities for a batch of texts efficiently.
    Returns a numpy array of shape (num_texts, 28) with probabilities for all 28 GoEmotions.

    Optimizations applied vs original:
    - Pre-tokenize all texts individually (no padding yet) — moves all Python-side
      tokenizer work outside the GPU loop so the GPU is never idle waiting for tokenization.
    - Custom collate_fn: pads each mini-batch only to the longest sequence IN that batch
      (dynamic padding), not the global max — reduces wasted FLOPs on padding tokens by
      3-4x for short lyric lines.
    - pin_memory=True: pre-pins CPU tensors in page-locked RAM for faster async DMA.
    - non_blocking=True: starts CPU→GPU transfer and returns immediately, overlapping
      data transfer with GPU compute from the previous batch.
    - autocast (FP16): runs model forward in half precision on Tensor Core GPUs (~1.5-2x).
    """
    if len(texts) == 0:
        return np.array([])

    # ── Step 1: Pre-tokenize each text individually (no padding) ──────────────
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    
    dataset_size = input_ids.shape[0]
    
    all_probs = []
    
    for i in range(0, dataset_size, batch_size):
    
        batch_input_ids = input_ids[i:i+batch_size].to(device, non_blocking=True)
        batch_attention = attention_mask[i:i+batch_size].to(device, non_blocking=True)
    
        inputs = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention
        }
    
        if USE_MIXED_PRECISION:
            with autocast():
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)
    
        probs = torch.nn.functional.softmax(outputs.logits.float(), dim=-1)
        all_probs.append(probs.cpu().numpy())
    
    return np.concatenate(all_probs, axis=0)


def map_to_fer(go_probs):
    """
    Map GoEmotions probability vectors (Nx28) to FER emotion vectors (Nx6).

    For each FER category, sum the GoEmotions probabilities belonging to that group:
        Angry_i    = sum of p_i^(e) for e in {anger, annoyance, disapproval, disgust}
        Fear_i     = sum of p_i^(e) for e in {fear, nervousness}
        Happy_i    = sum of p_i^(e) for e in {admiration, amusement, ..., relief}
        Sad_i      = sum of p_i^(e) for e in {sadness, grief, ..., embarrassment}
        Surprise_i = sum of p_i^(e) for e in {surprise, realization}
        Neutral_i  = sum of p_i^(e) for e in {neutral, curiosity, confusion}

    Args:
        go_probs: numpy array of shape (N, 28) — GoEmotions probabilities

    Returns:
        numpy array of shape (N, 6) — un-normalized FER scores
    """
    n = go_probs.shape[0]
    fer_scores = np.zeros((n, len(FER_LABELS)))

    for j, cat in enumerate(FER_LABELS):
        indices = FER_INDICES[cat]
        fer_scores[:, j] = go_probs[:, indices].sum(axis=1)

    return fer_scores


def normalize_fer(fer_scores):
    """
    Normalize FER emotion vectors so each row sums to 1.

    q_i = q*_i / sum_k(q*_i^(k))

    After normalization: Angry_i + Fear_i + Happy_i + Sad_i + Surprise_i + Neutral_i = 1

    Args:
        fer_scores: numpy array of shape (N, 6) — un-normalized FER scores

    Returns:
        numpy array of shape (N, 6) — normalized FER probabilities
    """
    row_sums = fer_scores.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)  # avoid division by zero
    return fer_scores / row_sums


def weighted_average_emotions(fer_vectors):
    """
    Aggregate line-level FER emotion vectors into a song-level vector using weighted averaging.

    Weight for each line: w_i = 1 - Neutral_i
    Song emotion vector:  E_song = sum(w_i * q_i) / sum(w_i)

    Args:
        fer_vectors: numpy array of shape (N, 6) — normalized FER vectors for each line

    Returns:
        Aggregated emotion vector of shape (6,)
    """
    if len(fer_vectors) == 0:
        return np.zeros(len(FER_LABELS))

    neutral_col = FER_LABELS.index('Neutral')

    # w_i = 1 - Neutral_i
    weights = 1 - fer_vectors[:, neutral_col]
    weights = weights.reshape(-1, 1)
    total_weight = np.sum(weights)

    if total_weight == 0:
        # All lines fully neutral → return uniform distribution
        return np.ones(len(FER_LABELS)) / len(FER_LABELS)

    weighted_sum = np.sum(weights * fer_vectors, axis=0)
    return weighted_sum / total_weight


def process_songs_batch(lyrics_list, batch_size=2048):
    """
    Process multiple songs efficiently by batching all lines together.

    Pipeline per song:
      1. Split lyrics into lines
      2. Predict GoEmotions (28-dim) for each line
      3. Map to FER emotions (6-dim) by summing groups
      4. Normalize FER vector
      5. Compute weight w_i = 1 - Neutral_i
      6. Aggregate via weighted average → song-level FER vector
      7. Determine dominant emotion

    Args:
        lyrics_list: List of lyrics strings
        batch_size: Batch size for GPU processing

    Returns:
        List of dicts, each with 'fer_vector' (6,) and 'dominant_emotion' (str)
    """
    all_lines = []
    song_line_counts = []

    for lyrics in lyrics_list:
        if pd.isna(lyrics) or str(lyrics).strip() == "":
            song_line_counts.append(0)
            continue

        lines = [line.strip() for line in str(lyrics).split('\n') if line.strip()]
        all_lines.extend(lines)
        song_line_counts.append(len(lines))

    # Batch predict GoEmotions for all lines at once
    if len(all_lines) > 0:
        all_go_probs = predict_emotions_batch(all_lines, batch_size=batch_size)
        all_fer_scores = map_to_fer(all_go_probs)
        all_fer_normalized = normalize_fer(all_fer_scores)
    else:
        all_fer_normalized = np.array([]).reshape(0, len(FER_LABELS))

    # Aggregate emotions per song
    results = []
    line_idx = 0

    for line_count in song_line_counts:
        if line_count == 0:
            fer_vec = np.zeros(len(FER_LABELS))
            dominant = 'Neutral'
        else:
            song_fer = all_fer_normalized[line_idx:line_idx + line_count]
            fer_vec = weighted_average_emotions(song_fer)
            dominant = FER_LABELS[np.argmax(fer_vec)]
            line_idx += line_count

        results.append({'fer_vector': fer_vec, 'dominant_emotion': dominant})

    return results


## 6. Configure Dataset Path and Parameters

In [ ]:

# CONFIGURATION FOR LARGE DATASETS (3M rows)

# Dataset paths (Kaggle)
data_path = '/kaggle/input/datasets/ankit477/genius-dataset-cleaned/cleaned_lyrics.csv'
output_path = '/kaggle/working/lyrics_with_emotions.csv'

# Processing parameters
READ_CHUNK_SIZE = 50000  # Read 50K rows from CSV at a time (memory efficient)
PROCESS_CHUNK_SIZE = 5000  # Process 5000 songs together (for batch processing)
BATCH_SIZE = 2048  # GPU batch size for line processing (safe for 15GB VRAM; lower to 256 if OOM)
CLEANED_LYRICS_COLUMN = 'clean_lyrics'  # Column name in your CSV

# ── Chunk Range Configuration ─────────────────────────────────────────────────
# Specify which chunks to process (1-indexed)
# Set to 0 for automatic behavior (start from beginning or resume from existing file)
START_CHUNK = 10  # Start from this chunk (0 = auto-detect from existing file or start from 1)
END_CHUNK = 20    # Stop after this chunk (0 = process all remaining chunks)

# Examples:
#   START_CHUNK=1,  END_CHUNK=10  → Process chunks 1-10 only
#   START_CHUNK=11, END_CHUNK=20  → Process chunks 11-20 only
#   START_CHUNK=0,  END_CHUNK=0   → Process everything (auto-resume if file exists)
#   START_CHUNK=5,  END_CHUNK=0   → Start from chunk 5, process until end

# Get total row count without loading entire dataset
print("Counting total rows (using pandas to handle multi-line lyrics fields)...")
total_rows = sum(len(chunk) for chunk in pd.read_csv(data_path, chunksize=READ_CHUNK_SIZE))
print(f"Total rows counted: {total_rows:,}")
total_chunks = (total_rows + READ_CHUNK_SIZE - 1) // READ_CHUNK_SIZE

# Check for existing progress (resume capability)
import os
already_processed = 0

if START_CHUNK > 0:
    # Manual start: begin from specified chunk
    already_processed = (START_CHUNK - 1) * READ_CHUNK_SIZE
    print(f"\n✓ Manual start set: Beginning from chunk {START_CHUNK} (row {already_processed + 1:,})")
elif os.path.exists(output_path):
    print("\nExisting output file found! Checking progress...")
    already_processed = sum(len(chunk) for chunk in pd.read_csv(output_path, chunksize=READ_CHUNK_SIZE))
    detected_chunk = (already_processed + READ_CHUNK_SIZE - 1) // READ_CHUNK_SIZE
    print(f"  Already processed: {already_processed:,} songs ({detected_chunk} chunks)")
    print(f"  Remaining: {total_rows - already_processed:,} songs")
    print(f"\n✓ Will RESUME from chunk {detected_chunk + 1}")
else:
    print("\nNo existing output file. Starting fresh from chunk 1.")

# Display chunk range info
effective_start_chunk = max(1, (already_processed // READ_CHUNK_SIZE) + 1)
effective_end_chunk = END_CHUNK if END_CHUNK > 0 else total_chunks

print(f"\n{'='*60}")
print(f"Chunk Range Configuration:")
print(f"  Total chunks in dataset: {total_chunks}")
print(f"  Start chunk: {effective_start_chunk}")
print(f"  End chunk: {effective_end_chunk}")
if END_CHUNK > 0:
    chunks_to_process = max(0, effective_end_chunk - effective_start_chunk + 1)
    print(f"  Chunks to process: {chunks_to_process}")
    print(f"  Rows to process: ~{chunks_to_process * READ_CHUNK_SIZE:,}")
else:
    print(f"  Chunks to process: {effective_end_chunk - effective_start_chunk + 1} (all remaining)")

print(f"\nDataset Info:")
print(f"  Total songs: {total_rows:,}")
print(f"  Read chunk size: {READ_CHUNK_SIZE:,} rows")
print(f"  Process chunk size: {PROCESS_CHUNK_SIZE} songs")
print(f"  GPU batch size: {BATCH_SIZE}")
print(f"\nConfiguration complete!")


Counting total rows (using pandas to handle multi-line lyrics fields)...


### Output Dataset Structure

The saved CSV file will contain **8 columns**:

1. **`cleaned_lyrics`** - Original cleaned lyrics from your input dataset
2. **6 FER emotion columns** - Normalized probability scores (sum to 1.0):
   - Angry, Fear, Happy, Sad, Surprise, Neutral
3. **`dominant_emotion`** - The emotion with the highest score for each song

**Mapping**: The 28 GoEmotions are grouped into 6 FER categories:
- **Angry** ← anger, annoyance, disapproval, disgust
- **Fear** ← fear, nervousness
- **Happy** ← admiration, amusement, approval, caring, desire, excitement, gratitude, joy, love, optimism, pride, relief
- **Sad** ← sadness, grief, disappointment, remorse, embarrassment
- **Surprise** ← surprise, realization
- **Neutral** ← neutral, curiosity, confusion

## 7. Process Dataset in Chunks and Save Incrementally
**Memory-efficient processing:** Reads CSV in chunks, processes with GPU batching, saves results incrementally

**Resume capability:** If Kaggle session terminates, simply re-run this cell to resume from where it left off!

In [ ]:

import time
from tqdm.auto import tqdm as tqdm_auto  # auto-selects notebook widget vs terminal bar

# Determine where to start (resume from existing progress)
rows_to_skip = already_processed if already_processed > 0 else 0
remaining_rows = total_rows - rows_to_skip

# Initialize CSV reader with chunking, skipping already processed rows
if rows_to_skip > 0:
    # Skip already processed rows
    csv_reader = pd.read_csv(data_path, chunksize=READ_CHUNK_SIZE, skiprows=range(1, rows_to_skip + 1))
    append_mode = True
else:
    csv_reader = pd.read_csv(data_path, chunksize=READ_CHUNK_SIZE)
    append_mode = False

# Calculate chunks for progress bar
remaining_chunks = (remaining_rows + READ_CHUNK_SIZE - 1) // READ_CHUNK_SIZE
total_read_chunks = (total_rows + READ_CHUNK_SIZE - 1) // READ_CHUNK_SIZE
starting_chunk = (rows_to_skip + READ_CHUNK_SIZE - 1) // READ_CHUNK_SIZE if rows_to_skip > 0 else 0

processed_songs = rows_to_skip
start_time = time.time()

effective_start_chunk = starting_chunk + 1
effective_end_chunk = END_CHUNK if END_CHUNK > 0 else total_read_chunks

if rows_to_skip > 0:
    print(f"\nRESUMING processing from song {rows_to_skip + 1:,}...")
    print(f"Progress: {rows_to_skip:,}/{total_rows:,} ({rows_to_skip/total_rows*100:.1f}%)")
else:
    print(f"\nStarting processing of {total_rows:,} songs...")

print(f"Processing chunks {effective_start_chunk} to {effective_end_chunk}")
print(f"Results will be saved to: {output_path}")
print(f"Output columns: cleaned_lyrics + {FER_LABELS} + dominant_emotion")
print(f"\n{'='*60}")

# tqdm progress bar — renders as an updating widget in Jupyter, a single updating
pbar = tqdm_auto(
    csv_reader,
    total=remaining_chunks,
    desc=f"Chunk {effective_start_chunk}/{total_read_chunks}",
    unit="chunk",
    position=0,
)

# Process each chunk from CSV
for chunk_idx, df_chunk in enumerate(pbar):
    chunk_start_time = time.time()
    actual_chunk_num = starting_chunk + chunk_idx + 1

    # ── Check if we should stop at this chunk ────────────────────────────────
    if END_CHUNK > 0 and actual_chunk_num > END_CHUNK:
        print(f"\n{'='*60}")
        print(f"✓ STOPPED after chunk {END_CHUNK} as requested")
        print(f"  Processed {processed_songs:,} songs total")
        print(f"\nTo continue processing:")
        print(f"  Set START_CHUNK = {END_CHUNK + 1}")
        print(f"  Set END_CHUNK = {END_CHUNK + 10} (or 0 for all remaining)")
        break

    # Verify column exists (only on first chunk)
    if chunk_idx == 0:
        if CLEANED_LYRICS_COLUMN not in df_chunk.columns:
            pbar.close()
            print(f"\nError: Column '{CLEANED_LYRICS_COLUMN}' not found!")
            print(f"Available columns: {df_chunk.columns.tolist()}")
            raise ValueError(f"Please update CLEANED_LYRICS_COLUMN")

    # Get lyrics from this chunk
    chunk_lyrics = df_chunk[CLEANED_LYRICS_COLUMN].tolist()

    # Calculate total number of inner batches for this chunk
    num_batches = (len(chunk_lyrics) + PROCESS_CHUNK_SIZE - 1) // PROCESS_CHUNK_SIZE

    # Inner progress bar — tracks batch-level progress within the current chunk
    inner_pbar = tqdm_auto(
        range(0, len(chunk_lyrics), PROCESS_CHUNK_SIZE),
        total=num_batches,
        desc=f"  Batch 0/{num_batches}",
        unit="batch",
        leave=False,
        position=1,
    )

    # Process songs in smaller batches for GPU efficiency
    chunk_results = []
    for batch_idx, batch_start in enumerate(inner_pbar):
        batch_end = min(batch_start + PROCESS_CHUNK_SIZE, len(chunk_lyrics))
        batch_lyrics = chunk_lyrics[batch_start:batch_end]
        batch_results = process_songs_batch(batch_lyrics, batch_size=BATCH_SIZE)
        chunk_results.extend(batch_results)

        # Update inner bar description and stats after each batch
        inner_pbar.set_description(f"  Batch {batch_idx + 1}/{num_batches}")
        inner_pbar.set_postfix(
            songs=f"{batch_end:,}/{len(chunk_lyrics):,}",
            pct=f"{batch_end / len(chunk_lyrics) * 100:.1f}%",
        )

    inner_pbar.close()

    # Convert to DataFrame with FER emotion columns + dominant_emotion
    fer_vectors = np.array([r['fer_vector'] for r in chunk_results])
    dominant_emotions = [r['dominant_emotion'] for r in chunk_results]

    emotion_df = pd.DataFrame(fer_vectors, columns=FER_LABELS)

    result_chunk = pd.DataFrame()
    result_chunk['cleaned_lyrics'] = df_chunk[CLEANED_LYRICS_COLUMN].values
    result_chunk = pd.concat([result_chunk, emotion_df], axis=1)
    result_chunk['dominant_emotion'] = dominant_emotions

    # Save to CSV incrementally (append if resuming)
    if append_mode:
        result_chunk.to_csv(output_path, index=False, mode='a', header=False)
        os.sync()
    else:
        result_chunk.to_csv(output_path, index=False, mode='w')
        append_mode = True  # Switch to append mode after first write

    processed_songs += len(df_chunk)
    chunk_time = time.time() - chunk_start_time
    songs_per_sec = len(df_chunk) / chunk_time if chunk_time > 0 else 0
    progress_pct = (processed_songs / total_rows) * 100

    # Update progress bar description and stats in-place (single updating line)
    pbar.set_description(f"Chunk {actual_chunk_num}/{total_read_chunks}")
    pbar.set_postfix(
        songs=f"{processed_songs:,}/{total_rows:,}",
        pct=f"{progress_pct:.1f}%",
        speed=f"{songs_per_sec:.0f}/s",
    )

    # Clear memory (do NOT call torch.cuda.empty_cache() here — it forces PyTorch's
    del df_chunk, chunk_lyrics, chunk_results, fer_vectors, dominant_emotions, emotion_df, result_chunk

pbar.close()

# Release cached GPU memory once, at the very end (not per-chunk)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Final statistics
elapsed_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"\nProcessing Complete!")
print(f"  Total songs processed: {processed_songs:,}")
print(f"  Total time: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"  Speed: {processed_songs/elapsed_time:.1f} songs/second")
print(f"  Output saved to: {output_path}")
print(f"  Columns: cleaned_lyrics, {', '.join(FER_LABELS)}, dominant_emotion")


---

## What if Kaggle Session Disconnects?
**To Resume:**
- Reconnect to Kaggle
- Run Cell 1-5 (setup cells)
- Run Cell 6 and change the value of already processed chunk
- Run Cell 7 (will resume from the last processed song)

**Note:** The progress bar will show overall completion percentage, so you can track how much is left!